In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
project_root = Path().resolve()  
sys.path.append(str(project_root / "libs/MIA-synthetic-main"))

import os
os.environ["SYNTHCITY_DEVICE"] = "cpu"
import random

import torch
torch.device("cpu")

import tapas
from tapas.attacks.set_classifiers import RandomTargetedQueryFeature
from utils.my_attack import MyAttack
from utils.benchmark_pipeline import BenchmarkPipeline
from utils.benchmark_metric import AUCMetric
from utils.tapas.tapas_data_processors import AdultDataProcessor
from utils.tapas.query_based_attack import RandomTargetedQueryFeatureFast
from utils.basegenerators import BenchmarkGenerator, ReprosynGenerator, Raw
from utils.customgenerators import SynthcityGenerator, SDVGenerator
#from utils.privpgdgenerator import PrivpgdGenerator 

# some example generators 
from reprosyn.methods import DS_PRIVBAYES
from reprosyn.methods import DS_BAYNET
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.single_table import CTGANSynthesizer
from sdv.single_table import TVAESynthesizer

In [ ]:
# define generator, some examples: 
rep_priv = ReprosynGenerator(DS_PRIVBAYES, label="PrivBayes", epsilon=1.0, histogram_bins = 10, degree=1)
rep_bay = ReprosynGenerator(DS_BAYNET, label="BayNet", histogram_bins=10, degree=1)  

syn_tab = SynthcityGenerator("ddpm", label="TabDDPM", 
                    lr=0.0001,
                    batch_size=64,
                    n_iter=200,
                    num_timesteps=100,
                    model_params={
                        'n_layers_hidden': 4,
                        'n_units_hidden':64,
                        'dropout': 0.0008
                    })
syn_tvae = SynthcityGenerator("tvae", label="tvae")
syn_samp = SynthcityGenerator("uniform_sampler", label="uniform_sampler")
syn_bay = SynthcityGenerator("bayesian_network", label="bayesian_network")
syn_great = SynthcityGenerator("great", label = "GReaT",
    device= "cpu",
    n_iter= 5,
    batch_size= 4,
    sampling_patience= 100
)
syn_rtvae = SynthcityGenerator("rtvae", label= "rtvae")

sdv_gauss = SDVGenerator(GaussianCopulaSynthesizer, label="GaussianCopulaSynthesizer")
sdv_tvae = SDVGenerator(TVAESynthesizer, label="TVAESynthesizer", epochs = 50)
sdv_ctgan = SDVGenerator(CTGANSynthesizer, label="CTGANSynthesizer", epochs = 50, batch_size=128, pac=2)

In [ ]:
# read in data and sample target record 
data = tapas.datasets.TabularDataset.read("data/adult")
data = AdultDataProcessor.process_tapas_tabulardataset(data)
ind = [int(random.random() * len(data.data))]
target_record = data.get_records(ind)
data.drop_records(ind, in_place=True)

# define attack 
attack = MyAttack(100, features=RandomTargetedQueryFeatureFast(target_record, low_order = 1,
                                                               high_order = 15, num_queries=10000))

# define pipeline
pipeline = BenchmarkPipeline(data, attack, [Raw(), syn_great,syn_tab,syn_rtvae,syn_samp,syn_tvae], 
                             target_record=target_record, data_size = 15000, size_of_datasets=1000)

# run pipeline with custom complexities, number of runs of full attack per complexity level, 
# number of attack models, custom metrics and specifying whether to use pre-generated data or newly generate
pipeline.run([10,50,100,500,1000,2000], 10, 1000, plot_style = "all", benchmarking_metric = AUCMetric(),
             classification_target_col="income", dcr_metric="gower", generate_data=True)